In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Gold — Modelagem Dimensional (Esquema Estrela) do Airbnb Rio de Janeiro
# MAGIC |---|---|
# MAGIC | **Autor** | João Pedro Paciello |
# MAGIC **Objetivo desta etapa:** modelar os dados limpos da Silver em um Esquema Estrela,
# MAGIC otimizado para responder diretamente às 5 perguntas de negócio do MVP, e documentar
# MAGIC cada tabela/coluna via `COMMENT`, alimentando o Catálogo de Dados do Unity Catalog
# MAGIC de forma versionada.
# MAGIC
# MAGIC **Desenho do modelo:**
# MAGIC - **Grão da fato:** um registro por imóvel (`id`) por mês (`data_referencia`) — mesmo grão da Silver.
# MAGIC - **`fato_diarias_airbnb`**: métricas que variam no tempo (preço, avaliações, disponibilidade,
# MAGIC   status de superhost no mês).
# MAGIC - **`dim_localizacao`**: bairro, grupo de bairro, cidade, coordenadas médias.
# MAGIC - **`dim_imovel`**: características do imóvel (tipo, capacidade, quartos, banheiros) —
# MAGIC   tratada como pouco variável no tempo (snapshot mais recente por imóvel).
# MAGIC - **`dim_host`**: dados cadastrais do host (nome, desde quando, tempo/taxa de resposta).
# MAGIC - **`dim_tempo`**: ano, mês, trimestre e estação do ano (hemisfério sul) — essencial
# MAGIC   para a pergunta de sazonalidade.

# COMMAND ----------

from pyspark.sql.functions import (
    col, avg, first, last, max as spark_max, row_number, month, year,
    quarter, when, concat, lit, lpad, trim, initcap, coalesce
)
from pyspark.sql.window import Window

CATALOGO = "mvp_airbnb_rj"
SCHEMA_SILVER = "silver"
TABELA_SILVER = "listings_silver"
SCHEMA_GOLD = "gold"

df_silver = spark.table(f"{CATALOGO}.{SCHEMA_SILVER}.{TABELA_SILVER}")
print(f"Registros na Silver: {df_silver.count()}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA_GOLD}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Achado de qualidade de dados: padronização de `city`
# MAGIC **Tratamento aplicado:**
# MAGIC - `city`: removidos espaços extras e padronizada a capitalização (`trim` + `initcap`),
# MAGIC   para que variações de escrita do mesmo nome não sejam tratadas como cidades diferentes.
# MAGIC - `neighbourhood_group_cleansed`: nulos substituídos por um valor sentinela
# MAGIC   `"Não informado"`, e a coluna passou a ser um **atributo descritivo** da dimensão,
# MAGIC   em vez de fazer parte da chave de agrupamento/junção (evitando o problema do JOIN).

# COMMAND ----------

df_silver = (
    df_silver
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn(
        "neighbourhood_group_cleansed",
        coalesce(trim(initcap(col("neighbourhood_group_cleansed"))), lit("Não informado"))
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. dim_localizacao
# MAGIC
# MAGIC Grão: um registro por combinação única de bairro + cidade (já com `city` padronizado).
# MAGIC `neighbourhood_group_cleansed` entra como atributo descritivo (primeiro valor não-nulo
# MAGIC encontrado no grupo), não como parte da chave — evitando o problema de JOIN com coluna
# MAGIC majoritariamente nula. A latitude/longitude médias servem apenas como referência
# MAGIC aproximada para visualizações em mapa — não substituem a coordenada exata de cada imóvel.

# COMMAND ----------

dim_localizacao = (
    df_silver
    .groupBy("neighbourhood_cleansed", "city")
    .agg(
        avg("latitude").alias("latitude_media"),
        avg("longitude").alias("longitude_media"),
        first("neighbourhood_group_cleansed", ignorenulls=True).alias("neighbourhood_group_cleansed"),
    )
)

# Chave substituta (surrogate key), gerada de forma determinística via row_number
# em vez de monotonically_increasing_id() — assim, se este notebook for reexecutado,
# a mesma linha sempre recebe a mesma chave (importante para consistência entre execuções).
janela_localizacao = Window.orderBy("neighbourhood_cleansed", "city")
dim_localizacao = dim_localizacao.withColumn("sk_localizacao", row_number().over(janela_localizacao))

print(f"Total de bairros/localizações distintas: {dim_localizacao.count()}")
display(dim_localizacao.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. dim_imovel
# MAGIC
# MAGIC Grão: um registro por imóvel (`id`). Características físicas do imóvel raramente
# MAGIC mudam mês a mês, então tratamos como dimensão "quase estática": para cada `id`,
# MAGIC pegamos o snapshot mais recente disponível (maior `data_referencia`).

# COMMAND ----------

janela_imovel_mais_recente = Window.partitionBy("id").orderBy(col("data_referencia").desc())

dim_imovel = (
    df_silver
    .withColumn("linha_mais_recente", row_number().over(janela_imovel_mais_recente))
    .filter(col("linha_mais_recente") == 1)
    .select("id", "property_type", "room_type", "accommodates", "bathrooms", "bedrooms", "beds")
    .withColumnRenamed("id", "sk_imovel")  # aqui a chave natural (id do listing) já serve como chave substituta
)

print(f"Total de imóveis distintos: {dim_imovel.count()}")
display(dim_imovel.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. dim_host
# MAGIC
# MAGIC Grão: um registro por host (`host_id`). Assim como em dim_imovel, usamos o snapshot
# MAGIC mais recente por host. 

# COMMAND ----------

janela_host_mais_recente = Window.partitionBy("host_id").orderBy(col("data_referencia").desc())

dim_host = (
    df_silver
    .withColumn("linha_mais_recente", row_number().over(janela_host_mais_recente))
    .filter(col("linha_mais_recente") == 1)
    .select("host_id", "host_name", "host_since", "host_response_time",
            "host_response_rate", "host_listings_count")
    .withColumnRenamed("host_id", "sk_host")

)

print(f"Total de hosts distintos: {dim_host.count()}")
display(dim_host.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. dim_tempo
# MAGIC
# MAGIC Grão: um registro por mês/ano distinto presente nos dados. Inclui a estação do ano
# MAGIC (considerando o hemisfério sul, já que o dataset é do Rio de Janeiro) — atributo
# MAGIC chave para responder a pergunta de sazonalidade do objetivo do MVP.

# COMMAND ----------

dim_tempo = df_silver.select("data_referencia").distinct()

dim_tempo = (
    dim_tempo
    .withColumn("ano", year("data_referencia"))
    .withColumn("mes", month("data_referencia"))
    .withColumn("trimestre", quarter("data_referencia"))
    .withColumn(
        "estacao_do_ano",
        # Hemisfério sul: verão (dez-fev), outono (mar-mai), inverno (jun-ago), primavera (set-nov)
        when(col("mes").isin(12, 1, 2), lit("Verão"))
        .when(col("mes").isin(3, 4, 5), lit("Outono"))
        .when(col("mes").isin(6, 7, 8), lit("Inverno"))
        .otherwise(lit("Primavera"))
    )
)

print(f"Total de meses distintos: {dim_tempo.count()}")
display(dim_tempo.orderBy("data_referencia"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. fato_diarias_airbnb
# MAGIC
# MAGIC Grão: um registro por imóvel (`id`) por mês (`data_referencia`) — igual à Silver.
# MAGIC Contém as métricas que efetivamente variam no tempo e que alimentam as 5 perguntas
# MAGIC de negócio: preço (+ flags de qualidade), avaliações, disponibilidade e status de superhost.

# COMMAND ----------

fato_diarias_airbnb = (
    df_silver
    .join(
        dim_localizacao.select("neighbourhood_cleansed", "city", "sk_localizacao"),
        on=["neighbourhood_cleansed", "city"],
        how="left"
    )
    .select(
        col("id").alias("sk_imovel"),
        col("host_id").alias("sk_host"),
        col("sk_localizacao"),
        col("data_referencia"),
        col("price"),
        col("flag_price_zero"),
        col("flag_price_outlier"),
        col("minimum_nights"),
        col("maximum_nights"),
        col("availability_30"),
        col("availability_60"),
        col("availability_90"),
        col("availability_365"),
        col("number_of_reviews"),
        col("review_scores_rating"),
        col("review_scores_accuracy"),
        col("review_scores_cleanliness"),
        col("review_scores_checkin"),
        col("review_scores_communication"),
        col("review_scores_location"),
        col("review_scores_value"),
        col("reviews_per_month"),
        col("host_is_superhost"),
        col("instant_bookable"),
    )
)

print(f"Total de registros na fato: {fato_diarias_airbnb.count()}")
display(fato_diarias_airbnb.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ### Validação defensiva — join com dim_localizacao
# MAGIC
# MAGIC **Boa prática:** Foi adicionado uma
# MAGIC checagem explícita do percentual de `sk_localizacao` nulo após o join. Um pequeno
# MAGIC percentual é esperado (registros cujo `neighbourhood_cleansed` já vinha nulo desde a
# MAGIC origem); um percentual alto indica que o join está quebrado novamente.

# COMMAND ----------

total_fato = fato_diarias_airbnb.count()
qtd_sk_localizacao_nula = fato_diarias_airbnb.filter(col("sk_localizacao").isNull()).count()
pct_sk_localizacao_nula = round((qtd_sk_localizacao_nula / total_fato) * 100, 2)

print(f"Registros com sk_localizacao nulo após o join: {qtd_sk_localizacao_nula} ({pct_sk_localizacao_nula}%)")

assert pct_sk_localizacao_nula < 10, (
    f"ALERTA: {pct_sk_localizacao_nula}% dos registros ficaram sem sk_localizacao após o join — "
    "percentual muito acima do esperado (nulos legítimos de neighbourhood_cleansed na origem "
    "giravam em torno de ~6%). Investigue se o join voltou a quebrar."
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Persistir as tabelas Delta na camada Gold

# COMMAND ----------

tabelas_gold = {
    "dim_localizacao": dim_localizacao,
    "dim_imovel": dim_imovel,
    "dim_host": dim_host,
    "dim_tempo": dim_tempo,
    "fato_diarias_airbnb": fato_diarias_airbnb,
}

for nome_tabela, df_tabela in tabelas_gold.items():
    try:
        (
            df_tabela
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{CATALOGO}.{SCHEMA_GOLD}.{nome_tabela}")
        )
        print(f"Tabela criada com sucesso: {CATALOGO}.{SCHEMA_GOLD}.{nome_tabela}")
    except Exception as erro:
        print(f"Falha ao gravar a tabela '{nome_tabela}': {erro}")
        raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Catálogo de Dados — documentando tabelas e colunas via COMMENT
# MAGIC
# MAGIC **Boa prática:** em vez de documentar o Catálogo de Dados só pela interface do Unity
# MAGIC Catalog (que não fica versionada no Git), foi aplicado os comentários diretamente via SQL
# MAGIC neste notebook.

# COMMAND ----------

# MAGIC %md
# MAGIC ### Nota de boa prática: escapando aspas simples em SQL
# MAGIC
# MAGIC O comando `COMMENT ON TABLE ... IS '...'` delimita o texto do comentário com aspas
# MAGIC simples. Se o próprio texto do comentário contiver uma aspas simples (ex: a palavra
# MAGIC `'Não informado'` citada dentro de uma explicação), o SQL interpreta essa aspas como o
# MAGIC FIM da string, quebrando a sintaxe do comando. A função abaixo escapa automaticamente
# MAGIC qualquer aspas simples nos textos, dobrando-a (`'` vira `''`), que é a forma padrão do
# MAGIC SQL de representar uma aspas literal dentro de uma string.

# COMMAND ----------

def escapar_aspas_sql(texto: str) -> str:
    """Escapa aspas simples em um texto para uso seguro dentro de uma string SQL."""
    return texto.replace("'", "''")

comentarios_tabelas = {
    "fato_diarias_airbnb": "Tabela fato: uma linha por imóvel por mês, com métricas de preço, avaliações, disponibilidade e status do host que variam ao longo do tempo. Grão: (sk_imovel, data_referencia).",
    "dim_localizacao": "Dimensão de localização: bairro, grupo de bairro e cidade, com coordenadas médias aproximadas para uso em mapas.",
    "dim_imovel": "Dimensão de imóvel: características físicas do imóvel (tipo, capacidade, quartos, banheiros), baseadas no snapshot mais recente de cada imóvel.",
    "dim_host": "Dimensão de host: dados cadastrais do anfitrião (nome, desde quando está na plataforma, tempo e taxa de resposta), baseados no snapshot mais recente.",
    "dim_tempo": "Dimensão de tempo: um registro por mês/ano presente no dataset, incluindo trimestre e estação do ano (hemisfério sul), usada para análises de sazonalidade.",
}

for nome_tabela, comentario in comentarios_tabelas.items():
    comentario_seguro = escapar_aspas_sql(comentario)
    spark.sql(f"COMMENT ON TABLE {CATALOGO}.{SCHEMA_GOLD}.{nome_tabela} IS '{comentario_seguro}'")

print("Comentários de tabela aplicados com sucesso.")

# COMMAND ----------

comentarios_colunas = {
    "fato_diarias_airbnb": {
        "sk_imovel": "Chave estrangeira para dim_imovel (id original do imóvel no Airbnb).",
        "sk_host": "Chave estrangeira para dim_host (id original do host no Airbnb).",
        "sk_localizacao": "Chave estrangeira para dim_localizacao.",
        "data_referencia": "Chave estrangeira para dim_tempo; primeiro dia do mês de referência da coleta.",
        "price": "Preço da diária em reais (BRL), já convertido de texto para decimal.",
        "flag_price_zero": "TRUE quando o preço da diária é R$ 0,00 — indica dado potencialmente inválido, a ser tratado na etapa de Análise.",
        "flag_price_outlier": "TRUE quando o preço está acima do percentil 99 da distribuição — outlier extremo, mantido mas sinalizado.",
        "number_of_reviews": "Quantidade total de avaliações recebidas pelo imóvel até o momento da coleta.",
        "review_scores_rating": "Nota geral de avaliação do imóvel (escala 0-100), pode ser nula se o imóvel ainda não possui avaliações.",
        "host_is_superhost": "Indica se o host tinha status de Superhost no mês de referência (status pode variar mês a mês).",
    },
    "dim_localizacao": {
        "sk_localizacao": "Chave substituta (surrogate key) da dimensão, gerada de forma determinística.",
        "neighbourhood_cleansed": "Nome do bairro, já normalizado pelo Inside Airbnb.",
        "neighbourhood_group_cleansed": "Agrupamento de bairros (zona/região) informado pela fonte; valor Não informado quando ausente na origem (a maioria dos registros deste dataset não possui esse campo preenchido).",
        "latitude_media": "Latitude média aproximada dos imóveis do bairro — uso apenas referencial para mapas.",
        "longitude_media": "Longitude média aproximada dos imóveis do bairro — uso apenas referencial para mapas.",
    },
    "dim_imovel": {
        "sk_imovel": "Chave substituta da dimensão (mesmo valor do id original do imóvel no Airbnb).",
        "property_type": "Tipo do imóvel (ex: Apartment, House, Loft).",
        "room_type": "Tipo de acomodação (ex: Entire home/apt, Private room, Shared room).",
        "accommodates": "Capacidade máxima de hóspedes do imóvel.",
    },
    "dim_host": {
        "sk_host": "Chave substituta da dimensão (mesmo valor do host_id original no Airbnb).",
        "host_since": "Data em que o host se cadastrou na plataforma.",
        "host_response_time": "Tempo médio de resposta do host às mensagens de hóspedes.",
    },
    "dim_tempo": {
        "data_referencia": "Primeiro dia do mês/ano de referência da coleta (chave da dimensão).",
        "estacao_do_ano": "Estação do ano no hemisfério sul (Verão, Outono, Inverno, Primavera), usada para análise de sazonalidade.",
    },
}

for nome_tabela, colunas in comentarios_colunas.items():
    for nome_coluna, comentario in colunas.items():
        comentario_seguro = escapar_aspas_sql(comentario)
        spark.sql(
            f"ALTER TABLE {CATALOGO}.{SCHEMA_GOLD}.{nome_tabela} "
            f"ALTER COLUMN {nome_coluna} COMMENT '{comentario_seguro}'"
        )

print("Comentários de coluna aplicados com sucesso.")
